# 09 — Final Test Evaluation

This is the **first and only** notebook in this project that reads the held-out test sets. Everything up to now — feature selection, preprocessing, model fitting, threshold selection — was decided using only training and validation data, specifically so that this final evaluation is honest.

This notebook:

- Loads the **already-fitted** pipelines saved by `05_model_development.ipynb` (no-`duration`, "realistic") and `06_leakage_experiment.ipynb` (with-`duration`, leakage diagnostic).
- Loads the **already-selected** thresholds from `results/frozen_evaluation_config.json` (`08_threshold_selection.ipynb`), chosen on validation data only.
- Evaluates every model **once**, on its matching saved test-split rows, at both the default 0.5 threshold and its frozen validation-selected threshold.

**No model is retrained, refit, or tuned here, and no threshold is adjusted based on anything seen in this notebook.** The with-`duration` models are leakage diagnostics only — included to quantify leakage on the test set, exactly as was done on validation, and explicitly **not** usable as a deployable pre-contact model, since `duration` isn't known before a call happens.

In [1]:
import os
import json

import numpy as np
import pandas as pd
from joblib import load

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    accuracy_score,
    confusion_matrix,
)

RESULTS_DIR = "../results"
MODELS_DIR = f"{RESULTS_DIR}/models"
FINAL_DIR = f"{RESULTS_DIR}/final"
os.makedirs(FINAL_DIR, exist_ok=True)

SPLITS = ["stratified", "chronological"]
POS_LABEL = "yes"

pd.set_option("display.max_columns", 100)

## Load the dataset, saved test-split indices, frozen thresholds, and fitted pipelines (read-only)

Nothing here is fit — `load()` deserializes pipelines that were already fit in Phases 5/6, and we only ever call `.predict_proba()` on them below.

In [2]:
CSV_NAME = "bank-additional-full.csv"

try:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
except ImportError:
    pass  # not running on Colab

_candidates = [
    f"../data/{CSV_NAME}",
    f"data/{CSV_NAME}",
    f"/content/drive/MyDrive/bank-marketing-leakage-audit/data/{CSV_NAME}",
    f"/content/drive/MyDrive/{CSV_NAME}",
]
DATA_PATH = next((p for p in _candidates if os.path.exists(p)), None)

if DATA_PATH is None:
    try:
        from google.colab import files  # type: ignore

        print(f"'{CSV_NAME}' not found in: {_candidates}")
        uploaded = files.upload()
        DATA_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Could not find '{CSV_NAME}' in any of: {_candidates}")

df_raw = pd.read_csv(DATA_PATH, sep=";")

split_paths = {
    "stratified": f"{RESULTS_DIR}/split_stratified.csv",
    "chronological": f"{RESULTS_DIR}/split_chronological.csv",
}
config_path = f"{RESULTS_DIR}/frozen_evaluation_config.json"
for path in list(split_paths.values()) + [config_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found — run the earlier notebooks (03, 08) first.")

splits = {name: pd.read_csv(path) for name, path in split_paths.items()}
with open(config_path) as f:
    frozen_config = json.load(f)

frozen_thresholds = {
    (c["split_strategy"], c["model"], c["feature_set"]): c["selected_threshold"]
    for c in frozen_config["configurations"]
}

MODEL_FILES = {
    ("stratified", "majority_baseline", "no_duration"): "stratified__majority_baseline.joblib",
    ("stratified", "logistic_regression", "no_duration"): "stratified__logistic_regression.joblib",
    ("stratified", "random_forest", "no_duration"): "stratified__random_forest.joblib",
    ("stratified", "logistic_regression", "with_duration"): "stratified__logistic_regression__with_duration.joblib",
    ("stratified", "random_forest", "with_duration"): "stratified__random_forest__with_duration.joblib",
    ("chronological", "majority_baseline", "no_duration"): "chronological__majority_baseline.joblib",
    ("chronological", "logistic_regression", "no_duration"): "chronological__logistic_regression.joblib",
    ("chronological", "random_forest", "no_duration"): "chronological__random_forest.joblib",
    ("chronological", "logistic_regression", "with_duration"): "chronological__logistic_regression__with_duration.joblib",
    ("chronological", "random_forest", "with_duration"): "chronological__random_forest__with_duration.joblib",
}

fitted_pipelines = {}
for key, filename in MODEL_FILES.items():
    path = f"{MODELS_DIR}/{filename}"
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found — run 05_model_development.ipynb and 06_leakage_experiment.ipynb first.")
    fitted_pipelines[key] = load(path)  # deserialize only — never fit here

print(f"Dataset: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print(f"Loaded {len(fitted_pipelines)} fitted pipelines and {len(frozen_thresholds)} frozen thresholds.")

Mounted at /content/drive


FileNotFoundError: ../results/split_stratified.csv not found — run the earlier notebooks (03, 08) first.

## Rebuild the two feature-set working dataframes

Identical feature selection and `pdays` recoding used in Phases 3/5/6 — needed only to reconstruct the exact input columns each saved pipeline expects. No fitting happens here; these are just feature matrices to feed into `.predict_proba()`.

In [ ]:
PRIMARY_FEATURES = [
    "age", "job", "marital", "education", "default", "housing", "loan",
    "pdays", "previous", "poutcome",
    "contact", "month", "day_of_week",
]
TARGET = "y"


def build_working_dataset(include_duration):
    cols = PRIMARY_FEATURES + (["duration"] if include_duration else []) + [TARGET]
    d = df_raw[cols].copy()
    d["previously_contacted"] = (d["pdays"] != 999).astype(int)
    d["days_since_previous_contact"] = d["pdays"].replace(999, np.nan)
    d = d.drop(columns=["pdays"])

    numerical = ["age", "previous", "days_since_previous_contact", "previously_contacted"]
    if include_duration:
        numerical = numerical + ["duration"]
    categorical = ["job", "marital", "education", "default", "housing", "loan",
                   "poutcome", "contact", "month", "day_of_week"]
    return d, numerical + categorical


data_no_duration, feature_cols_no_duration = build_working_dataset(include_duration=False)
data_with_duration, feature_cols_with_duration = build_working_dataset(include_duration=True)

assert "duration" not in feature_cols_no_duration
assert "duration" in feature_cols_with_duration

DATA_BY_FEATURE_SET = {
    "no_duration": (data_no_duration, feature_cols_no_duration),
    "with_duration": (data_with_duration, feature_cols_with_duration),
}

print("no_duration feature columns:", len(feature_cols_no_duration))
print("with_duration feature columns:", len(feature_cols_with_duration))

## Evaluate every model once on its held-out test rows

For each (split strategy, model, feature set): slice the saved **test** rows, call `.predict_proba()` on the already-fitted pipeline (no `.fit()` anywhere here), and compute every requested metric at both threshold 0.5 and the frozen validation-selected threshold (where one exists — the majority-class baseline has no tuned threshold, since its probability output is constant regardless of threshold).

In [ ]:
MODELS_BY_FEATURE_SET = {
    "no_duration": ["majority_baseline", "logistic_regression", "random_forest"],
    "with_duration": ["logistic_regression", "random_forest"],
}

MODEL_CATEGORY = {
    "no_duration": "realistic_pre_contact_model",
    "with_duration": "LEAKAGE_DIAGNOSTIC_ONLY_NOT_DEPLOYABLE",
}


def positive_class_proba(fitted_pipeline, X):
    pos_idx = list(fitted_pipeline.classes_).index(POS_LABEL)
    return fitted_pipeline.predict_proba(X)[:, pos_idx]  # predict_proba only — never fit


def metrics_at_threshold(y_true_bin, proba, threshold):
    y_pred_bin = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1]).ravel()
    return {
        "precision": precision_score(y_true_bin, y_pred_bin, zero_division=0),
        "recall": recall_score(y_true_bin, y_pred_bin, zero_division=0),
        "f1": f1_score(y_true_bin, y_pred_bin, zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }, y_pred_bin


test_metric_rows = []
test_prediction_rows = []

for split_name in SPLITS:
    split_table = splits[split_name]
    test_idx = split_table.loc[split_table["split"] == "test", "row_index"]

    for feature_set, model_names in MODELS_BY_FEATURE_SET.items():
        data, feature_cols = DATA_BY_FEATURE_SET[feature_set]
        X_test = data.loc[test_idx, feature_cols]
        y_test = data.loc[test_idx, TARGET]
        y_true_bin = (y_test.values == POS_LABEL).astype(int)

        for model_name in model_names:
            key = (split_name, model_name, feature_set)
            pipeline = fitted_pipelines[key]

            proba = positive_class_proba(pipeline, X_test)  # predict_proba only — no fit

            frozen_threshold = frozen_thresholds.get(key)  # None for majority_baseline
            threshold_for_frozen = frozen_threshold if frozen_threshold is not None else 0.5

            metrics_default, y_pred_default = metrics_at_threshold(y_true_bin, proba, 0.5)
            metrics_frozen, y_pred_frozen = metrics_at_threshold(y_true_bin, proba, threshold_for_frozen)

            row = {
                "split_strategy": split_name,
                "model": model_name,
                "feature_set": feature_set,
                "model_category": MODEL_CATEGORY[feature_set],
                "n_test": int(len(test_idx)),
                "test_prevalence": float(y_true_bin.mean()),
                "average_precision": float(average_precision_score(y_true_bin, proba)),
                "roc_auc": float(roc_auc_score(y_true_bin, proba)) if len(set(y_true_bin)) > 1 else np.nan,
                "accuracy_at_0.5": float(accuracy_score(y_true_bin, (proba >= 0.5).astype(int))),
                "precision_at_0.5": metrics_default["precision"],
                "recall_at_0.5": metrics_default["recall"],
                "f1_at_0.5": metrics_default["f1"],
                "tn_at_0.5": metrics_default["tn"], "fp_at_0.5": metrics_default["fp"],
                "fn_at_0.5": metrics_default["fn"], "tp_at_0.5": metrics_default["tp"],
                "frozen_threshold": frozen_threshold,  # None/NaN for majority_baseline
                "precision_at_frozen_threshold": metrics_frozen["precision"],
                "recall_at_frozen_threshold": metrics_frozen["recall"],
                "f1_at_frozen_threshold": metrics_frozen["f1"],
                "tn_at_frozen_threshold": metrics_frozen["tn"], "fp_at_frozen_threshold": metrics_frozen["fp"],
                "fn_at_frozen_threshold": metrics_frozen["fn"], "tp_at_frozen_threshold": metrics_frozen["tp"],
            }
            test_metric_rows.append(row)

            test_prediction_rows.append(pd.DataFrame({
                "row_index": test_idx.values,
                "split_strategy": split_name,
                "model": model_name,
                "feature_set": feature_set,
                "y_true": y_test.values,
                "y_proba_yes": proba,
                "y_pred_at_0.5": np.where(y_pred_default == 1, "yes", "no"),
                "y_pred_at_frozen_threshold": np.where(y_pred_frozen == 1, "yes", "no"),
            }))

            print(f"[{split_name:13s} | {feature_set:14s} | {model_name:20s}] "
                  f"AP={row['average_precision']:.4f}  ROC-AUC={row['roc_auc']:.4f}  "
                  f"F1@0.5={row['f1_at_0.5']:.4f}  F1@frozen={row['f1_at_frozen_threshold']:.4f}")

test_metrics_df = pd.DataFrame(test_metric_rows)
test_predictions_df = pd.concat(test_prediction_rows, ignore_index=True)

## Final test metrics table

One row per (split strategy, model, feature set) — 10 rows total. `model_category` makes the leakage-diagnostic models impossible to mistake for deployable ones at a glance.

In [ ]:
column_order = [
    "split_strategy", "model", "feature_set", "model_category",
    "n_test", "test_prevalence", "average_precision", "roc_auc", "accuracy_at_0.5",
    "precision_at_0.5", "recall_at_0.5", "f1_at_0.5",
    "tn_at_0.5", "fp_at_0.5", "fn_at_0.5", "tp_at_0.5",
    "frozen_threshold", "precision_at_frozen_threshold", "recall_at_frozen_threshold", "f1_at_frozen_threshold",
    "tn_at_frozen_threshold", "fp_at_frozen_threshold", "fn_at_frozen_threshold", "tp_at_frozen_threshold",
]
test_metrics_df = test_metrics_df[column_order].sort_values(["split_strategy", "feature_set", "model"]).reset_index(drop=True)
test_metrics_df

## Change in AP and ROC-AUC caused by `duration` — on the test set

The same before/after comparison done on validation in `06_leakage_experiment.ipynb` / `07_depth_analysis.ipynb`, now computed on the held-out test set, for Logistic Regression and Random Forest in both splits.

In [ ]:
no_dur = test_metrics_df[test_metrics_df["feature_set"] == "no_duration"].set_index(["split_strategy", "model"])
with_dur = test_metrics_df[test_metrics_df["feature_set"] == "with_duration"].set_index(["split_strategy", "model"])

leakage_test_delta_rows = []
for split_name in SPLITS:
    for model_name in ["logistic_regression", "random_forest"]:
        key = (split_name, model_name)
        ap_no = no_dur.loc[key, "average_precision"]
        ap_with = with_dur.loc[key, "average_precision"]
        auc_no = no_dur.loc[key, "roc_auc"]
        auc_with = with_dur.loc[key, "roc_auc"]
        leakage_test_delta_rows.append({
            "split_strategy": split_name,
            "model": model_name,
            "average_precision_no_duration": ap_no,
            "average_precision_with_duration": ap_with,
            "delta_average_precision": ap_with - ap_no,
            "roc_auc_no_duration": auc_no,
            "roc_auc_with_duration": auc_with,
            "delta_roc_auc": auc_with - auc_no,
        })

leakage_test_delta_df = pd.DataFrame(leakage_test_delta_rows)
leakage_test_delta_df

## Save final metrics and predictions

New files only, in `results/final/`:

- `test_metrics.csv` — the 10-row table above.
- `test_predictions.csv` — per-row test predictions/probabilities for every (split, model, feature_set) combination.
- `leakage_test_delta.csv` — the ΔAP / ΔROC-AUC table above.
- `README.md` — the leakage-diagnostic labeling warning, in plain text, next to the data.

Nothing in `results/` outside `results/final/` is written to.

In [ ]:
metrics_path = f"{FINAL_DIR}/test_metrics.csv"
predictions_path = f"{FINAL_DIR}/test_predictions.csv"
delta_path = f"{FINAL_DIR}/leakage_test_delta.csv"
readme_path = f"{FINAL_DIR}/README.md"

test_metrics_df.to_csv(metrics_path, index=False)
test_predictions_df.to_csv(predictions_path, index=False)
leakage_test_delta_df.to_csv(delta_path, index=False)

with open(readme_path, "w") as f:
    f.write(
        "# Final Test Evaluation — Model Labeling\n\n"
        "- `model_category == \"realistic_pre_contact_model\"`: the no-`duration` models "
        "(`majority_baseline`, `logistic_regression`, `random_forest`). These use only features "
        "known before a scheduled contact and are the only ones that could realistically be deployed.\n\n"
        "- `model_category == \"LEAKAGE_DIAGNOSTIC_ONLY_NOT_DEPLOYABLE\"`: the with-`duration` "
        "`logistic_regression` / `random_forest` models. `duration` is only known after a call ends, "
        "so these models' apparently strong performance is a measurement of leakage, not a usable "
        "pre-contact predictor. **Do not deploy these.**\n"
    )

print("Saved:", metrics_path, "-", test_metrics_df.shape)
print("Saved:", predictions_path, "-", test_predictions_df.shape)
print("Saved:", delta_path, "-", leakage_test_delta_df.shape)
print("Saved:", readme_path)

## Summary

- Evaluated all 10 (split × model × feature_set) combinations — including the majority-class baseline — on their saved **test** rows, using only already-fitted pipelines (`joblib.load`, never `.fit()`) and the frozen validation-selected thresholds (`08_threshold_selection.ipynb`).
- Reported AP, ROC-AUC, accuracy, precision/recall/F1 at both 0.5 and the frozen threshold, full confusion matrices, and test prevalence for every combination.
- Quantified ΔAP and ΔROC-AUC caused by `duration` on the test set, mirroring the validation-set leakage analysis.
- Every with-`duration` row is explicitly tagged `model_category = "LEAKAGE_DIAGNOSTIC_ONLY_NOT_DEPLOYABLE"` in both the CSV and an accompanying `README.md`, to prevent it from ever being mistaken for a deployable model.
- Saved `results/final/test_metrics.csv`, `results/final/test_predictions.csv`, `results/final/leakage_test_delta.csv`, and `results/final/README.md`.

This is the only notebook that reads the test set, and it does so exactly once, after every modeling and threshold decision had already been locked in from training/validation data alone. No existing artifact outside `results/final/` was modified.